In [1]:
import sys
import numpy as np
import time

from firm3d.field.boozermagneticfield import (
    BoozerRadialInterpolant,
    InterpolatedBoozerField,
)
from firm3d.field.tracing import (
    trace_particles_boozer,
    MaxToroidalFluxStoppingCriterion,
)
from firm3d.field.tracing_helpers import (
    initialize_position_profile,
    initialize_velocity_uniform,
)
from firm3d.util.constants import (
    ALPHA_PARTICLE_MASS,
    ALPHA_PARTICLE_CHARGE,
    FUSION_ALPHA_PARTICLE_ENERGY,
)
from firm3d.util.functions import proc0_print
from firm3d._core.util import parallel_loop_bounds

from firm3d.field.tracing_helpers import (
    initialize_position_uniform_vol,
    initialize_velocity_uniform,
)

from orbit_classification_Jens import OrbitClassification

from firm3d.util.mpi import comm_size, comm_world, verbose


try:
    from mpi4py import MPI

    comm = MPI.COMM_WORLD
    verbose = comm.rank == 0
    comm_size = comm.size
except ImportError:
    comm = None
    verbose = True
    comm_size = 1


# Bcs
Bc_res = 30
# modBmin_global = 5.515158152204317+0.5
# modBmax_global = 7.375629705430016-0.5
modBmin_global = 6.05
modBmax_global = 6.8
Bcs = np.linspace(modBmin_global,modBmax_global,Bc_res)


resolution = 48  # Resolution for field interpolation
nParticles = int(1e4)  # Number of particles to trace
reltol = 1e-8  # Relative tolerance for the ODE solver
abstol = 1e-8  # Absolute tolerance for the ODE solver
order = 3  # Order for radial interpolation
degree = 3  # Degree for 3d interpolation
# boozmn_filename = "/pscratch/sd/j/jensvdl/equil/boozmn/boozmn_div-opt_98_DESC_fixed_NS1024-single.nc"
boozmn_filename = "/Users/paullab/codes/firm3d_fork_10132025/firm3d/examples/inputs/boozmn_equil_G1600_DESC_fixed.nc"
tmax = 1e-2  # Time for integration
ns_interp = resolution
ntheta_interp = resolution
nzeta_interp = resolution
helicity_M = 1 # Helicity of the field strength, used to distinguish ripple and barely-trapped orbits
helicity_N = 0
dt_save = 1e-7  # Time interval for saving trajectory points

sys.stdout = open(f"stdout_{nParticles}_{resolution}_{comm_size}.txt", "a", buffering=1)

## Setup radial interpolation
bri = BoozerRadialInterpolant(boozmn_filename, order, no_K=True, comm=comm)

## Setup 3d interpolation
field = InterpolatedBoozerField(
    bri,
    degree,
    ns_interp=ns_interp,
    ntheta_interp=ntheta_interp,
    nzeta_interp=nzeta_interp,
)

# Define fusion birth distribution
# Bader, A., et al. "Modeling of energetic particle transport in optimized stellarators." Nuclear Fusion 61.11 (2021): 116060.
# nD = lambda s: (1 - s**5)  # Normalized density
# nT = nD
# T = lambda s: 11.5 * (1 - s)  # Temperature in keV

# # D-T cross-section
# def sigmav(T):
#     if T > 0:
#         return T ** (-2 / 3) * np.exp(-19.94 * T ** (-1 / 3))
#     else:
#         return 0

# # Reactivity profile
# reactivity = lambda s: nD(s) * nT(s) * sigmav(T(s))

# points_init = initialize_position_profile(field, nParticles, reactivity, comm=comm, seed=0)


In [ ]:
points = initialize_position_uniform_vol(field, nParticles,comm=comm_world)
print('here')
field.set_points(points)
print('here1')
# print(field.modB().shape)
B = field.modB()[:,0] # B at each point. cool

Ekin = FUSION_ALPHA_PARTICLE_ENERGY
mass = ALPHA_PARTICLE_MASS
charge = ALPHA_PARTICLE_CHARGE
# Initialize uniformly distributed parallel velocities
vpar0 = np.sqrt(2 * Ekin / mass)
# vpar_init = initialize_velocity_uniform(vpar0, nParticles, comm=comm, seed=0)
print('here2')

def delta_rho(s): # returns the total drho for one particle
    # s = np.array(poinc.s_all)
    s_max = np.max(s)
    s_min = np.min(s)
    drho = s_max**(0.5)-s_min**(0.5) # one for each s,eta initialization
    return np.sum(drho)

for Bc in Bcs:
    print('Bc = ',Bc)
    filter_i=[]
    points_filtered=[]
    B_filtered=[]
    i=0
    for _B in B:
        if _B>=Bc: # not trapped
            # filter_i.append(i)
            continue
        else:
            points_filtered.append(points[i,:])
            B_filtered.append(_B)
        i+=1
    points_filtered = np.array(points_filtered) # := (num particles,coord)
    nParticles_filtered = len(points_filtered[:,0])
    B_filtered = np.array(B_filtered)
    vpar_init = np.random.choice([-1, 1], size=nParticles_filtered) * vpar0 * np.sqrt(1 - B_filtered/Bc)


    first, last = parallel_loop_bounds(comm, nParticles_filtered)
    for iParticle in range(first, last):
        point = np.zeros((1, 3))
        point[0,:] = points_filtered[iParticle, :]
        ## Trace alpha particles in Boozer coordinates until they hit the s = 1 surface
        res_tys, res_hits = trace_particles_boozer(
            field,
            point,
            [vpar_init[iParticle]],
            tmax=tmax,
            mass=mass,
            charge=charge,
            Ekin=Ekin,
            vpars=[0],
            vpars_stop=False,
            stopping_criteria=[MaxToroidalFluxStoppingCriterion(1.0)],
            forget_exact_path=False,
            abstol=abstol,
            reltol=reltol,
            dt_save=dt_save,
        )
        print('passed tracing')
        res_hit = res_hits[0]
        res_ty = res_tys[0]
        bounce_times = []
        if len(res_hit) > 0:
            if np.any(res_hit[:,1]==-1): # Particle was lost to the wall
                np.savetxt('test/Bc'+str(Bc)+'particle_' + str(iParticle) + '_traj.txt', res_ty)
                np.savetxt('test/Bc'+str(Bc)+'particle_' + str(iParticle) + '_hits.txt', res_hit)
            else:
                continue # Particle was not lost to the wall, skip

            oc = OrbitClassification(field, Ekin, mass, charge, helicity_M, helicity_N)
            particle_dict = oc.classify_orbit(res_ty, res_hit)



            np.savez(f"test/Bc"+str(Bc)+"particle_{iParticle}.npz", **particle_dict)